# 01 — Nettoyage des données et construction de T et E

**Projet Horizon : décrochage et tutorat** — tâches DA-01 à DA-07, jalon M1 (données propres).

Ce notebook relance le nettoyage des 5 fichiers bruts et la construction de la base d'analyse, puis présente les résultats.
**Attention : l'exécuter réécrit `data/processed/` et `docs/journal_nettoyage.md`** (le résultat est identique à chaque exécution).
Toute la logique est dans `src/` ; ce notebook ne fait qu'appeler ces fonctions, afficher et interpréter.

- Règles appliquées et leur justification : `docs/decisions.md` (décisions validées par Saliou) ;
- détail de chaque traitement, avec les nombres de lignes : `docs/journal_nettoyage.md` ;
- anomalies constatées sur les fichiers bruts : `docs/audit_qualite.md`.

In [1]:
import re
import sys
from pathlib import Path

import pandas as pd

# Le notebook est dans notebooks/ : on remonte jusqu'à la racine du dépôt (dossier qui
# contient src/config.py) pour pouvoir importer src/, quel que soit le dossier de lancement.
racine = Path.cwd()
while not (racine / "src" / "config.py").exists():
    racine = racine.parent
sys.path.insert(0, str(racine))

from src import base_survie, config
from src.nettoyage import lancer_nettoyage

pd.set_option("display.max_columns", 30)

## 1. Relancer le nettoyage et la construction de la base

Le nettoyage lit `data/raw/` (jamais modifié) et écrit `data/processed/` ; la construction de la base écrit `base_survie.csv` et `base_periode_personne.csv`.

In [2]:
resultats = lancer_nettoyage()
base, periode = base_survie.lancer()

etudiants : 5085 lignes -> 5000 lignes
  regle_entetes : 0 lignes touchées
  regle_identifiants : 25 lignes touchées
  regle_modalites : 1719 lignes touchées
  regle_nombres : 509 lignes touchées
  regle_valeurs_impossibles : 57 lignes touchées
  regle_doublons : 85 lignes touchées
  regle_mention_note : 196 lignes touchées
inscriptions : 5024 lignes -> 5000 lignes
  regle_identifiants : 0 lignes touchées
  regle_statuts : 400 lignes touchées
  regle_dates : 1516 lignes touchées
  regle_doublons : 20 lignes touchées
  regle_dates_inversees : 12 lignes touchées
  regle_orphelins : 14 lignes touchées
  regle_controle_rentree : 0 lignes touchées
  regle_inscriptions_reconstituees : 10 lignes touchées
  regle_sortie_et_drapeaux : 68 lignes touchées
activite_hebdo : 130967 lignes -> 129783 lignes
  regle_identifiants : 1310 lignes touchées
  regle_nombres : 129683 lignes touchées
  regle_semaines_invalides : 70 lignes touchées
  regle_valeurs_impossibles : 112 lignes touchées
  regle_notes_

base_survie : 5000 étudiants, E = 1 pour 1440
période-personne : 121938 lignes


## 2. Tableau avant / après des 5 tables

In [3]:
avant_apres = pd.DataFrame([
    {
        "table": r["table"],
        "lignes avant": r["avant"]["lignes"],
        "lignes après": r["apres"]["lignes"],
        "lignes retirées": r["avant"]["lignes"] - r["apres"]["lignes"],
        "doublons de clé avant": r["avant"]["doublons_id"],
        "doublons de clé après": r["apres"]["doublons_id"],
        "valeurs manquantes avant": sum(r["avant"]["manquants"].values()),
        "valeurs manquantes après (colonnes d'origine)": sum(
            r["apres"]["manquants"][c] for c in r["avant"]["manquants"] if c in r["apres"]["manquants"]),
    }
    for r in resultats
])
display(avant_apres)
total_avant, total_apres = avant_apres["lignes avant"].sum(), avant_apres["lignes après"].sum()
print(f"Sur les 5 tables, {total_avant} lignes brutes deviennent {total_apres} lignes propres "
      f"({total_avant - total_apres} lignes retirées, surtout des doublons). "
      f"Après nettoyage, il ne reste aucun doublon : chaque étudiant (ou chaque semaine d'un étudiant) "
      f"n'apparaît qu'une fois.")

,table,lignes avant,lignes après,lignes retirées,doublons de clé avant,doublons de clé après,valeurs manquantes avant,valeurs manquantes après (colonnes d'origine)
0,etudiants,5085,5000,85,85,0,522,417
1,inscriptions,5024,5000,24,20,0,3931,3923
2,activite_hebdo,130967,129783,1184,1060,0,74872,74288
3,tutorat,1291,1255,36,30,0,8,30
4,messages,3200,3175,25,0,0,18,0


Sur les 5 tables, 145567 lignes brutes deviennent 144213 lignes propres (1354 lignes retirées, surtout des doublons). Après nettoyage, il ne reste aucun doublon : chaque étudiant (ou chaque semaine d'un étudiant) n'apparaît qu'une fois.


In [4]:
rejets = pd.DataFrame([
    {"fichier de rejets": f"data/processed/rejets/{fichier}", "lignes": nb}
    for r in resultats for fichier, nb in r.get("rejets", {}).items()
])
display(rejets)
print(f"{rejets['lignes'].sum()} lignes ont été retirées des tables mais conservées dans des fichiers de rejets "
      "(inscriptions et tutorats d'étudiants inconnus, semaines impossibles, activité après la sortie, "
      "messages vides) : rien n'est effacé sans trace.")

,fichier de rejets,lignes
0,data/processed/rejets/inscriptions_sans_etudia...,14
1,data/processed/rejets/activite_semaine_invalid...,70
2,data/processed/rejets/activite_apres_sortie.csv,54
3,data/processed/rejets/tutorat_sans_etudiant.csv,6
4,data/processed/rejets/messages_vides.csv,25


169 lignes ont été retirées des tables mais conservées dans des fichiers de rejets (inscriptions et tutorats d'étudiants inconnus, semaines impossibles, activité après la sortie, messages vides) : rien n'est effacé sans trace.


## 3. Décisions appliquées

In [5]:
texte_decisions = (racine / "docs" / "decisions.md").read_text(encoding=config.ENCODAGE)
decisions = re.findall(r"^## (D-\d+)", texte_decisions, flags=re.MULTILINE)
validees = re.findall(r"^- Statut : validée", texte_decisions, flags=re.MULTILINE)
nb_traitements = sum(len(r["entrees"]) for r in resultats)
print(f"Le registre compte {len(decisions)} décisions, dont {len(validees)} validées par Saliou ; "
      f"le journal de nettoyage contient {nb_traitements} lignes de traitement. "
      "Chaque correction des données renvoie à une décision écrite avant d'être appliquée "
      "(voir docs/decisions.md et docs/journal_nettoyage.md).")

Le registre compte 44 décisions, dont 44 validées par Saliou ; le journal de nettoyage contient 63 lignes de traitement. Chaque correction des données renvoie à une décision écrite avant d'être appliquée (voir docs/decisions.md et docs/journal_nettoyage.md).


## 4. Événements : décrochage (E = 1) et censure (E = 0)

Définitions imposées (cahier §4.3) : décrochage = abandon déclaré **ou** 4 semaines consécutives sans connexion ni devoir rendu ; censure = transfert, ou étudiant toujours là à la semaine 30.

In [6]:
repartition = (base["type_evenement"].value_counts().rename("effectif").to_frame())
repartition["%"] = (100 * repartition["effectif"] / len(base)).round(1)
repartition["E"] = [1 if t in config.TYPES_AVEC_EVENEMENT else 0 for t in repartition.index]
display(repartition)
nb_e = int(base["E"].sum())
silencieux = int((base["type_evenement"] == config.EVENEMENT_DECROCHAGE_SILENCIEUX).sum())
print(f"{nb_e} étudiants sur {len(base)} ont décroché pendant les 30 semaines, soit {100 * nb_e / len(base):.1f} %. "
      f"Parmi eux, {silencieux} ({100 * silencieux / nb_e:.0f} %) n'ont jamais déclaré leur abandon : "
      "ils ont simplement cessé de se connecter et de rendre leurs devoirs. "
      "Sans la règle des 4 semaines, ces décrochages seraient invisibles.")

,effectif,%,E
type_evenement,,,
fin_de_suivi,3441,68.8,0
abandon_declare,1004,20.1,1
decrochage_silencieux,408,8.2,1
transfert,117,2.3,0
abandon_sans_date,22,0.4,1
abandon_date_invalide,6,0.1,1
perdu_de_vue,2,0.0,0


1440 étudiants sur 5000 ont décroché pendant les 30 semaines, soit 28.8 %. Parmi eux, 408 (28 %) n'ont jamais déclaré leur abandon : ils ont simplement cessé de se connecter et de rendre leurs devoirs. Sans la règle des 4 semaines, ces décrochages seraient invisibles.


In [7]:
par_cohorte = base.groupby("cohorte").agg(etudiants=("E", "size"), decrochages=("E", "sum"))
par_cohorte["taux de décrochage (%)"] = (100 * par_cohorte["decrochages"] / par_cohorte["etudiants"]).round(1)
display(par_cohorte)
print(f"Le taux de décrochage est presque identique d'une cohorte à l'autre "
      f"({par_cohorte['taux de décrochage (%)'].min()} % à {par_cohorte['taux de décrochage (%)'].max()} %). "
      f"La durée moyenne de suivi est de {base['T'].mean():.2f} semaines sur 30.")

,etudiants,decrochages,taux de décrochage (%)
cohorte,,,
2024,2495,715,28.7
2025,2505,725,28.9


Le taux de décrochage est presque identique d'une cohorte à l'autre (28.7 % à 28.9 %). La durée moyenne de suivi est de 24.39 semaines sur 30.


## 5. Sensibilité : seuil d'inactivité et semaines « inconnues »

In [8]:
tables = base_survie.lire_tables()
activite = base_survie.ajouter_etats(tables["activite_hebdo"])
autre_lecture = base_survie.construire_t_e(activite, tables["inscriptions"],
                                           lecture_inconnu=config.LECTURE_INCONNU_SENSIBILITE)
autre_lecture = autre_lecture.set_index("id_etudiant").reindex(base["id_etudiant"])
reference = base.set_index("id_etudiant")
changes = int(((autre_lecture["T"] != reference["T"]) | (autre_lecture["E"] != reference["E"])).sum())

sensibilite = pd.DataFrame([
    {"variante": "seuil 3 semaines", "décrochages (E = 1)": int(base["E_seuil3"].sum())},
    {"variante": "seuil 4 semaines (référence)", "décrochages (E = 1)": nb_e},
    {"variante": "seuil 5 semaines", "décrochages (E = 1)": int(base["E_seuil5"].sum())},
    {"variante": "seuil 4, semaines inconnues comptées inactives", "décrochages (E = 1)": int(autre_lecture["E"].sum())},
])
sensibilite["écart avec la référence"] = sensibilite["décrochages (E = 1)"] - nb_e
display(sensibilite)
ecart_max = sensibilite["écart avec la référence"].abs().max()
print(f"Changer le seuil de 3 à 5 semaines, ou compter comme inactives les semaines où une donnée manque, "
      f"change le nombre de décrochages d'au plus {ecart_max} étudiants sur {nb_e} "
      f"(moins de {100 * ecart_max / nb_e:.1f} %) ; la seconde variante change T ou E pour {changes} étudiants. "
      "La définition du décrochage est donc robuste : les résultats ne dépendent presque pas de ces choix.")

,variante,décrochages (E = 1),écart avec la référence
0,seuil 3 semaines,1445,5
1,seuil 4 semaines (référence),1440,0
2,seuil 5 semaines,1435,-5
3,"seuil 4, semaines inconnues comptées inactives",1441,1


Changer le seuil de 3 à 5 semaines, ou compter comme inactives les semaines où une donnée manque, change le nombre de décrochages d'au plus 5 étudiants sur 1440 (moins de 0.3 %) ; la seconde variante change T ou E pour 20 étudiants. La définition du décrochage est donc robuste : les résultats ne dépendent presque pas de ces choix.


## 6. Tutorat

`tutore` = 1 si le tutorat a commencé **strictement avant** la semaine T (D-37) : un tutorat qui démarre la semaine de l'événement, ou après, n'a pas pu l'empêcher. Les 5 tutorats à semaine inconnue sont comptés tutorés ; leur traitement sera décidé dans la partie causale (CA-01).

> On ne compare pas ici le taux de décrochage des tutorés et des non-tutorés : cette comparaison naïve serait biaisée (biais de sélection et biais du temps immortel). L'effet du tutorat sera estimé dans la partie causale.

In [9]:
statuts = base["statut_tutorat"].value_counts().rename("effectif").to_frame()
statuts["%"] = (100 * statuts["effectif"] / len(base)).round(1)
display(statuts)
nb_tutores = int(base["tutore"].sum())
ecartes = int(base["statut_tutorat"].isin([config.STATUT_TUTORAT_MEME_SEMAINE, config.STATUT_TUTORAT_APRES_T]).sum())
print(f"{nb_tutores} étudiants ({100 * nb_tutores / len(base):.1f} %) ont commencé un tutorat avant leur semaine T. "
      f"{ecartes} autres n'ont commencé le tutorat que la semaine de leur événement (décrochage ou censure) ou après : "
      "ils ne comptent pas comme tutorés, car ce tutorat n'a pas pu agir sur leur parcours.")

,effectif,%
statut_tutorat,,
non_tutore,3745,74.9
tutore,1199,24.0
tutorat_meme_semaine_que_T,31,0.6
tutorat_apres_T,20,0.4
tutorat_semaine_inconnue,5,0.1


1204 étudiants (24.1 %) ont commencé un tutorat avant leur semaine T. 51 autres n'ont commencé le tutorat que la semaine de leur événement (décrochage ou censure) ou après : ils ne comptent pas comme tutorés, car ce tutorat n'a pas pu agir sur leur parcours.


## 7. Format période-personne

In [10]:
resume_periode = pd.Series({
    "lignes (étudiant × semaine, de 1 à T)": len(periode),
    "lignes avec un événement": int(periode["evenement"].sum()),
    "lignes avec tutorat commencé": int((periode["tutorat_commence"] == 1).sum()),
    "lignes à tutorat de semaine inconnue (vide)": int(periode["tutorat_commence"].isna().sum()),
}).rename("valeur").to_frame()
display(resume_periode)
print(f"Le format période-personne compte une ligne par étudiant et par semaine de suivi ({len(periode)} lignes). "
      "Il garde l'activité semaine par semaine sans la résumer sur tout le suivi : on évite ainsi d'utiliser "
      "une information qui dépend du décrochage lui-même (D-36).")

,valeur
"lignes (étudiant × semaine, de 1 à T)",121938
lignes avec un événement,1440
lignes avec tutorat commencé,24119
lignes à tutorat de semaine inconnue (vide),127


Le format période-personne compte une ligne par étudiant et par semaine de suivi (121938 lignes). Il garde l'activité semaine par semaine sans la résumer sur tout le suivi : on évite ainsi d'utiliser une information qui dépend du décrochage lui-même (D-36).
